# Lab 6, Task 2B: Experimental Receiver Diversity

Use the SIMO field in the dual-channel capture to compare SC, EGC, and MRC with measured channel coefficients.

In [1]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display, Image

def find_lab_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "support" / "lab6_dsp.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the Lab 6 folder tree")

LAB_ROOT = find_lab_root()
sys.path.insert(0, str(LAB_ROOT / "support"))
import lab6_dsp as lab
import lab6_tools as tools
print("Lab folder:", LAB_ROOT.name)


Lab folder: Lab6


## Load one acquired frame

In [2]:
capture_dir = LAB_ROOT/'work/captures'
summary = __import__('json').loads((capture_dir/'lab6_measurement_summary.json').read_text())
tx_gains_db = tuple(int(value) for value in summary['experiment']['tx_gains_db'])
gain = tx_gains_db[1]
waveform = lab.build_waveform()
rx0 = lab.load_complex64(capture_dir/f'lab6_rx_tx{gain}_ch0_c64.dat', lab.CAPTURE_SAMPLES)
rx1 = lab.load_complex64(capture_dir/f'lab6_rx_tx{gain}_ch1_c64.dat', lab.CAPTURE_SAMPLES)
noise0 = lab.load_complex64(capture_dir/'lab6_noise_ch0_c64.dat')[:250000]
noise1 = lab.load_complex64(capture_dir/'lab6_noise_ch1_c64.dat')[:250000]
if noise0.size < 250000 or noise1.size < 250000:
    raise ValueError('Each noise record must contain at least 250000 samples.')
capture = lab.analyze_hardware_capture(rx0,rx1,noise0,noise1,waveform)
frame = capture['analyzed_frames'][0]
r = frame['payload_symbols_rx']
h = frame['simo_channel']
print('SIMO channel:',h)

FileNotFoundError: /Users/marcpham/jupyter_notebook_files/Embedded-Wireless-Design-Labs/Lab6/work/captures/lab6_rx_tx45_ch0_c64.dat

## Combine the two measured branches

`r` contains two rows of received QPSK symbols. `h` contains one complex flat-fading estimate per receive branch.

In [ ]:
h_matrix = np.repeat(h[:,None],r.shape[1],axis=1)
# STUDENT: implement normalized SC, EGC, and MRC. Divide by lab.DIGITAL_PEAK after combining.
best = ...
sc = ...
egc_weights = ...
egc = ...
mrc = ...
outputs={'sc':sc,'egc':egc,'mrc':mrc}
for name,z in outputs.items():
    errors=np.count_nonzero(lab.qpsk_detect(z)!=waveform.payload_bits)
    print(name.upper(),'errors',errors,'BER',errors/waveform.payload_bits.size,'EVM',lab.rms_evm(z,waveform.payload_symbols))

## Plotting supplied

<span style='color:#1464b4'><b>Report:</b> Include both branch constellations, all three combined constellations, BER or error counts, EVM.</span>

In [ ]:
out = LAB_ROOT/'work/figures/task2_student_diversity_outputs.png'
tools.plot_diversity_outputs(outputs,out)
display(Image(filename=str(out),width=1000))

<span style='color:#1464b4'><b>Report:</b> Include a received-quality and received power
comparison across TX gain.</span>

<span style='color:#1464b4'><b>Report:</b> Compare measured trends with the Task 1B simulation.</span>

The measured diversity results generally followed the same trends observed in the Task 1B simulation. In both cases, increasing signal quality improved performance, and the diversity combiners outperformed selection combining. The simulation showed that MRC achieved the lowest BER across the SNR range, followed by EGC and then SC. The experimental measurements showed a similar ordering, with MRC and EGC producing lower RMS EVM than SC at all TX gains. As TX gain increased from 40 dB to 50 dB, the measured EVM decreased for all combining methods, indicating improved reception quality. While the measured BER values were nearly identical across the tested gains and combiners, likely due to the relatively high-SNR measurement conditions, the EVM trends were consistent with the simulation and confirmed the expected advantage of MRC and EGC over SC.

<span style='color:#1464b4'><b>Report:</b> Explain when diversity is preferable to
spatial multiplexing.</span>

Diversity is preferred when the goal is reliable communication rather than maximum throughput. By combining multiple received copies of the same signal, diversity reduces fading and improves BER and EVM. Spatial multiplexing increases data rate by transmitting multiple streams simultaneously but requires stronger channel conditions and good separation between spatial channels.